# Inspeção e isolamento de falhas em redes industriais
**MECAI — ICMC-USP | Busca em largura e busca em profundidade com Python e orientação a objetos**

**Trabalho individual. Prazo sugerido: 10 dias a partir da disponibilização pelo professor.**

Nome: ____________________  Número USP: ____________________

Uma fábrica mantém sensores, controladores e gateways conectados por enlaces de comunicação. Você desenvolverá um serviço que encontra rotas com o menor número de enlaces, identifica componentes isoladas e ciclos e avalia interdições de conexões. Os dados são sintéticos e não descrevem uma instalação real.

A entrega é **este notebook preenchido e executado**, incluindo código, testes próprios, gráficos e análise. Estimativa: 14–18 horas. Não é uma atividade para acompanhar durante a aula. Implemente em Python usando as classes exigidas, depois reinicie a sessão e execute tudo em ordem antes de entregar.


## 1. Objetivos, escopo e avaliação

As aulas anexas são a base: BFS com fila FIFO e marcação ao inserir; camadas, distâncias e predecessores; DFS recursiva; DFS iterativa com quadros; estados N/E/F; descoberta, término, floresta, ciclo não dirigido e complexidade.

A rede é um **grafo simples, não dirigido e sem pesos**. Um enlace conta como um salto. Uma rota com poucos saltos não implica menor latência, maior capacidade, segurança física ou maior confiabilidade. Não implemente Dijkstra nem ordenação topológica nesta atividade.

| Critério | Peso |
|---|---:|
| Correção de BFS, DFS recursiva e DFS iterativa | 35% |
| Orientação a objetos efetiva e estruturas de apoio | 25% |
| Serviço industrial e análise de interdições | 15% |
| Testes, casos especiais e métricas | 10% |
| Análise experimental e recomendação técnica | 15% |

**OO é obrigatória:** herança do contrato abstrato, composição de estratégias no serviço, estado encapsulado no grafo e nas estruturas, resultados como objetos e troca da DFS sem modificar o serviço. Os testes verificam delegação observável e independência. A revisão do professor verificará se os algoritmos estão realmente nos métodos das classes. Apenas colocar chamadas a funções globais de BFS/DFS dentro de uma classe não atende ao requisito.

Não use NetworkX, igraph ou outras implementações prontas de percurso, componentes, caminhos mínimos ou ciclos nas classes avaliadas. `list`, `dict`, `set`, `tuple`, `dataclass` e `collections.deque` são permitidos. A fila deve encapsular `deque`; a pilha deve encapsular uma lista. Não use `list.pop(0)` como fila. As referências independentes fornecidas são exclusivas da infraestrutura de validação.

Sugestão de organização: dias 1–2 modelo e estruturas; 3–4 BFS; 5–6 duas DFS; 7 serviço e testes; 8–9 experimentos; 10 revisão e entrega.


## 2. Ambiente e infraestrutura fornecida

Use um ambiente Python 3 no Colab, sem GPU. A atividade usa a biblioteca padrão, pandas e matplotlib. Execute a célula abaixo. Nenhum arquivo externo, login ou conexão com Drive é necessário.

`Metricas`, `Evento` e `EstrategiaPercurso` são fornecidas para padronizar a integração. As células de implementação posteriores estão vazias. Os programas que você precisa desenvolver não estão nas células de apoio.


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from collections import deque
from time import perf_counter
import math
import random
import statistics
import pandas as pd
import matplotlib.pyplot as plt

@dataclass(frozen=True)
class Metricas:
    vertices_descobertos: int
    entradas_examinadas: int
    insercoes: int
    remocoes: int
    max_fronteira: int

@dataclass(frozen=True)
class Evento:
    tipo: str
    vertice: str
    fronteira: tuple
    estados: tuple
    tempo: int | None

class EstrategiaPercurso(ABC):
    @abstractmethod
    def executar(self, grafo, origem=None):
        """BFS exige origem. DFS percorre a floresta inteira, priorizando origem quando informada."""
        raise NotImplementedError

ARESTAS_AULA_BFS = (('A','B'),('A','C'),('B','D'),('B','E'),('C','E'),
                   ('C','F'),('D','G'),('E','G'),('E','H'),('F','H'))
ARESTAS_AULA_DFS = (('A','B'),('A','C'),('B','D'),('B','E'),('D','E'),
                   ('C','F'),('C','G'),('F','H'))
CLASSES_EXIGIDAS = ('QuadroDFS','Grafo','Fila','PilhaQuadros','ResultadoBFS','ResultadoDFS',
                   'BuscaLargura','BuscaProfundidadeRecursiva','BuscaProfundidadeIterativa','ServicoInspecao')

def pronto():
    return all(nome in globals() for nome in CLASSES_EXIGIDAS)

print('Ambiente pronto. Implemente as classes antes de executar testes e experimentos.')


Ambiente pronto. Implemente as classes antes de executar testes e experimentos.


## 3. Exercício A — Grafo, fila e pilha como objetos

Implemente as classes da tabela na célula vazia a seguir. Não mantenha estruturas mutáveis em atributos de classe nem use estado global para armazenar uma rede.

| Classe | Interface obrigatória |
|---|---|
| `Grafo` | `Grafo(vertices, arestas)`; propriedades `vertices` e `arestas`; `vizinhos(u)`; `sem_arestas(removidas)` |
| `Fila` | `Fila()`; `inserir(valor)`; `remover()`; `vazia()`; `__len__()`; `fotografia()` |
| `QuadroDFS` | `QuadroDFS(vertice, proximo=0)` com atributos mutáveis `vertice` e `proximo` |
| `PilhaQuadros` | `PilhaQuadros()`; `empilhar(quadro)`; `desempilhar()`; `topo()`; `vazia()`; `__len__()`; `fotografia()` |

### Contrato do grafo

- Vértices são strings não vazias e únicas. Os extremos de cada aresta devem estar entre os vértices informados.
- Rejeite vértices duplicados, laços e arestas repetidas, inclusive `(A,B)` seguida de `(B,A)`, com `ValueError`.
- `vertices` retorna uma tupla em ordem alfabética; `arestas` retorna uma tupla ordenada de pares canônicos `(menor, maior)`; `vizinhos(u)` retorna uma tupla em ordem alfabética. Essa ordem é a usada nos percursos.
- O grafo não muda após a construção. Não exponha listas internas que permitam alterar sua adjacência.
- `vizinhos` de vértice desconhecido gera `KeyError`.
- `sem_arestas` devolve **outro grafo**, com os mesmos vértices e sem os enlaces indicados. Uma remoção de aresta inexistente gera `ValueError`. Não altere o original.

### Contrato da fila e da pilha

`Fila.remover` obedece FIFO. `PilhaQuadros.desempilhar` obedece LIFO. Remover de estrutura vazia e consultar topo vazio geram `IndexError`.

`topo()` devolve o **objeto QuadroDFS armazenado**, de modo que alterar seu `proximo` atualize a continuação na pilha. Já `fotografia()` devolve uma cópia lógica imutável: tupla de valores na fila, do início ao fim; tupla de pares `(vertice, proximo)` na pilha, da base ao topo. Uma fotografia antiga não pode mudar quando o índice de um quadro mudar.

Sugestão de classe parcial, apenas como orientação:

```python
@dataclass
class QuadroDFS:
    vertice: str
    proximo: int = 0
```

A ordenação na construção é pré-processamento e ficará fora das medições de percurso. Não ordene a lista de vizinhos a cada chamada de `vizinhos`.


**A1 — Encapsulamento e invariantes**

_Escreva sua resposta aqui._


## 4. Exercício B — Resultados e reconstrução do caminho

Implemente duas classes de resultado. Você pode usar `@dataclass(frozen=True)`. Os dicionários pertencem ao resultado e devem continuar válidos após outra execução da mesma estratégia. `frozen=True` impede reatribuir campos, mas não torna um dicionário internamente imutável: não compartilhe dicionários mutáveis entre execuções.

### ResultadoBFS

Construtor posicional:

```python
ResultadoBFS(origem, distancias, pais, ordem, metricas, eventos)
```

Atributos com esses nomes. `distancias` e `pais` incluem **todos** os vértices. Inalcançáveis recebem `math.inf` e `None`. A origem recebe distância zero e pai `None`. `ordem` é a tupla da ordem de descoberta. `metricas` é um objeto `Metricas`; `eventos` é uma tupla de `Evento`.

Implemente `caminho(destino)`: devolver a tupla da origem ao destino, `None` se o destino existe mas é inalcançável e `KeyError` se o destino não pertence ao grafo.

```text
se destino não está no dicionário de distâncias: erro
se distância[destino] é infinita: devolver None
caminho_inverso ← vazio
u ← destino
enquanto u não for nulo:
    acrescentar u ao caminho_inverso
    u ← pai[u]
devolver tupla com a ordem invertida
```

### ResultadoDFS

Construtor posicional:

```python
ResultadoDFS(pais, descoberta, termino, ordem_descoberta, ordem_termino,
             raizes, raiz_de, ciclo, metricas, eventos)
```

`descoberta` e `termino` são dicionários com os tempos; as ordens e raízes são tuplas. `raiz_de[v]` informa a raiz da árvore de DFS de v. Em uma rede não dirigida, cada árvore cobre uma componente conexa. `ciclo` é `None` ou uma tupla de vértices que certifica um ciclo.

Implemente a propriedade `componentes`: tupla de componentes, na ordem de `raizes`, cada uma como tupla de vértices. Agrupe em uma passagem pelos vértices, usando `raiz_de`. Não execute uma nova busca para produzir essa propriedade.


## 5. Métricas e rastro: convenções comuns

Em todos os algoritmos, N significa não descoberto, E em processamento e F finalizado. Use essas strings exatas nos eventos.

| Campo de Metricas | Regra de contagem |
|---|---|
| `vertices_descobertos` | Quantidade de transições N → E |
| `entradas_examinadas` | Somar 1 toda vez que examinar um vizinho em uma lista de adjacência, mesmo se ele já foi descoberto |
| `insercoes` | Enfileiramentos em BFS, chamadas iniciadas em DFS recursiva ou quadros empilhados em DFS iterativa |
| `remocoes` | Desenfileiramentos, chamadas encerradas ou quadros retirados, respectivamente |
| `max_fronteira` | Maior tamanho simultâneo da fila ou quantidade de chamadas/quadros ativos |

As métricas reiniciam em cada `executar`. Grafo vazio na DFS produz métricas todas zero. Uma DFS completa examina `2*len(grafo.arestas)` entradas, pois o grafo é não dirigido. Uma BFS pode examinar menos, se houver componentes inalcançáveis.

### Registro didático

Todas as estratégias aceitam `registrar=False` no construtor. Com `False`, `eventos` deve ser a tupla vazia. Com `True`, acrescente:

1. `Evento('descobrir', u, fronteira, estados, tempo)` imediatamente **depois** de inserir u na fila/pilha ou iniciar a chamada.
2. `Evento('finalizar', u, fronteira, estados, tempo)` depois de marcar u como F. Na DFS, faça a fotografia **depois** de retirar o quadro ou retornar da chamada.

`estados` é `tuple(estados.items())`, com o dicionário inicializado na ordem de `grafo.vertices`. Em BFS, `tempo=None`; em DFS, o relógio do evento é o valor corrente após a descoberta/finalização. `fronteira` segue o contrato de fotografia da estrutura.

Na DFS recursiva, você pode manter uma lista auxiliar de quadros **apenas para instrumentação**, atualizada com os índices do laço e os retornos. A exploração deve ocorrer pelas chamadas recursivas. Isso permite comparar seus eventos com os da versão iterativa.

**As fotografias completas adicionam tempo e memória**: são 2|V| eventos em uma DFS e cada evento copia estados e possivelmente uma pilha. Portanto o modo didático pode consumir O(|V|²). Desative o registro ao medir os percursos. As métricas de fronteira não medem bytes nem incluem o histórico de eventos.


## 6. Exercício C — Busca em largura e rota mínima

Implemente `BuscaLargura(EstrategiaPercurso)` com:

```python
__init__(self, registrar=False, fabrica_fila=Fila)
executar(self, grafo, origem=None) -> ResultadoBFS
```

Crie a fila chamando `self.fabrica_fila()`. Use os métodos da classe, sem acessar seus atributos privados. `origem=None` gera `ValueError`; origem desconhecida gera `KeyError`. BFS percorre somente a componente alcançável da origem, preservando os outros vértices com distância infinita.

```text
inicializar estados N, distâncias infinitas e pais nulos
marcar origem E; distância[origem] ← 0
enfileirar origem; registrar descoberta
enquanto fila não estiver vazia:
    u ← remover do início
    para v nos vizinhos de u, na ordem do grafo:
        contar uma entrada examinada
        se estado[v] = N:
            estado[v] ← E
            distância[v] ← distância[u] + 1
            pai[v] ← u
            enfileirar v; registrar descoberta
    estado[u] ← F; registrar finalização
devolver objeto ResultadoBFS
```

Inclua as métricas especificadas. **A marcação ocorre antes do enfileiramento** para impedir que dois vértices insiram o mesmo vizinho. No grafo da aula de largura, a rota de A até H deve ser `(A,B,E,H)`, com três enlaces.


**C1 — Por que BFS encontra uma rota mínima?**

_Escreva sua resposta aqui._


## 7. Exercício D — DFS recursiva, floresta e certificado de ciclo

Implemente `BuscaProfundidadeRecursiva(EstrategiaPercurso)` com:

```python
__init__(self, registrar=False)
executar(self, grafo, origem=None) -> ResultadoDFS
```

Esta DFS percorre **o grafo inteiro**. Se `origem` for informada, comece por ela e depois examine os demais vértices em ordem alfabética. Se ela não existir, gere `KeyError`. Com `origem=None`, examine todos em ordem alfabética. O relógio começa em zero **uma vez por execução**, não uma vez por componente.

```text
inicializar estados N, pais nulos, relógio zero e estruturas de resultado
para cada candidato a raiz na ordem definida:
    se ainda está N:
        registrar nova raiz
        VISITAR(candidato, candidato)

VISITAR(u, raiz):
    estado[u] ← E; avançar relógio e registrar d[u]
    raiz_de[u] ← raiz; registrar descoberta e início da chamada
    para v em adj[u], na ordem dada:
        avançar a continuação do quadro de u; contar entrada examinada
        se estado[v] = N:
            pai[v] ← u; VISITAR(v, raiz)
        senão, se estado[v] = E e v ≠ pai[u] e ainda não há certificado:
            construir o certificado do ciclo
    estado[u] ← F; avançar relógio e registrar f[u]
    encerrar chamada; registrar finalização
```

### Certificado do ciclo

O grafo é simples e não dirigido. A aresta para o pai não fecha um novo ciclo. Ao encontrar um vizinho E que não seja o pai, esse vizinho é um ancestral ativo. Siga os pais de u até esse ancestral, inverta o ramo e acrescente o ancestral novamente. Exemplo: `(B,D,E,B)`. Devolva somente o **primeiro** ciclo encontrado nessa regra. Não procure todos os ciclos.

Os vértices internos do certificado não se repetem, o primeiro é igual ao último e todos os enlaces do certificado existem no grafo. Para um grafo sem ciclos, devolva `None`. Você pode implementar um método privado de montagem do certificado e reaproveitar seu desenho na outra DFS.

**Nesta célula, a exploração deve realmente chamar um método ou função interna recursiva.** A lista auxiliar de rastro não pode substituir a recursão.


**D1 — Componentes, tempos e ciclo**

_Escreva sua resposta aqui._


## 8. Exercício E — DFS iterativa com quadros

Implemente `BuscaProfundidadeIterativa(EstrategiaPercurso)` com:

```python
__init__(self, registrar=False, fabrica_pilha=PilhaQuadros)
executar(self, grafo, origem=None) -> ResultadoDFS
```

Use `self.fabrica_pilha()` e os métodos públicos da pilha. Mantenha as mesmas convenções de raízes, pais, estados, relógio, ciclo, métricas e eventos da versão recursiva.

```text
inicializar uma vez os estados, pais e relógio
para cada candidato a raiz na ordem definida:
    se está N:
        registrar raiz; descobrir raiz; empilhar (raiz, 0)
        enquanto a pilha não estiver vazia:
            quadro ← consultar topo, sem remover
            u ← quadro.vertice; i ← quadro.proximo
            se i = quantidade de vizinhos de u:
                finalizar u; retirar quadro; registrar finalização
            senão:
                v ← adj[u][i]
                quadro.proximo ← i + 1
                contar uma entrada examinada
                se estado[v] = N:
                    pai[v] ← u; raiz_de[v] ← raiz
                    descobrir v; empilhar (v, 0); registrar descoberta
                senão, se v está E e não é pai[u] e não há certificado:
                    construir o certificado do ciclo
```

Avance o índice **antes** de empilhar o filho. A atualização precisa alterar o objeto armazenado na pilha. Não substitua esta solução pela variante que empilha somente vértices em ordem inversa: ela não conserva, por si só, as continuações e os términos exigidos.

A equivalência requerida inclui `pais`, `descoberta`, `termino`, `ordem_descoberta`, `ordem_termino`, `raizes`, `raiz_de`, `ciclo`, métricas e eventos. Nenhuma chamada recursiva é permitida nesta classe.


**E1 — Onde o laço continua?**

_Escreva sua resposta aqui._


## 9. Exercício F — Serviço de inspeção por composição

Implemente `ServicoInspecao(grafo, busca_largura, busca_profundidade)` com os atributos públicos de mesmo nome. As estratégias recebidas já são objetos. O serviço **não escolhe algoritmos por nome ou tipo**, não implementa os laços de busca e não cria uma estratégia para substituir silenciosamente a recebida.

| Método | Contrato |
|---|---|
| `rota(origem, destino)` | Delegar à estratégia de largura e a `ResultadoBFS.caminho`. Retornar tupla ou `None`; vértice inexistente gera `KeyError`. |
| `diagnosticar()` | Delegar à estratégia de profundidade sem origem e devolver seu `ResultadoDFS`. |
| `avaliar_interdicao(removidas, origem, destino)` | Criar outro grafo sem os enlaces indicados e comparar com o original, sempre usando as estratégias recebidas. |

O relatório da interdição é um dicionário com **exatamente** as chaves:

```text
rota_antes: tupla de caminho ou None
rota_depois: tupla de caminho ou None
novos_inacessiveis: tupla alfabética dos vértices antes alcançáveis e agora inalcançáveis
componentes_antes: quantidade de componentes no grafo original
componentes_depois: quantidade de componentes no grafo alterado
```

Execute BFS antes e depois para avaliar a alcançabilidade e DFS antes e depois para contar componentes. Um vértice que já era inalcançável não é um novo afetado. As estratégias devem poder executar novamente sem alterar resultados já devolvidos.

Teste a substituição de `BuscaProfundidadeRecursiva` por `BuscaProfundidadeIterativa` sem modificar o código do serviço. Não é necessário detectar pontos de articulação nem resolver um problema de otimização de interdições.


**F1 — Por que usar composição?**

_Escreva sua resposta aqui._


## 10. Testes fornecidos e referências independentes

A próxima célula testa grafos vazios, vértices isolados, redes desconexas, ciclos, origem inválida, FIFO/LIFO, tempos, pais, conservação do grafo, execuções repetidas e delegação OO. As referências usam Floyd–Warshall em grafos pequenos para distâncias e união de conjuntos para componentes. Assim, as duas DFS não validam uma à outra como única evidência.

Os testes são públicos e não constituem prova completa de correção ou de orientação a objetos. Não os remova para esconder falhas. Laços sem progresso podem exigir interromper a execução no Colab. A infraestrutura informa quando as classes ainda não foram implementadas.


In [ ]:
def referencias_independentes(g):
    """Floyd–Warshall para distâncias e união de conjuntos para componentes.
    Uso somente em testes pequenos; não é uma implementação de BFS ou DFS.
    """
    vs = g.vertices
    dist = {(u, v): (0 if u == v else math.inf) for u in vs for v in vs}
    pai = {u: u for u in vs}
    def raiz(u):
        while pai[u] != u:
            pai[u] = pai[pai[u]]; u = pai[u]
        return u
    for u, v in g.arestas:
        dist[u, v] = dist[v, u] = 1
        pai[raiz(u)] = raiz(v)
    for k in vs:
        for u in vs:
            for v in vs:
                dist[u, v] = min(dist[u, v], dist[u, k] + dist[k, v])
    grupos = {}
    for u in vs: grupos.setdefault(raiz(u), set()).add(u)
    comps = {frozenset(c) for c in grupos.values()}
    tem_ciclo = len(g.arestas) > len(vs) - len(comps)
    return dist, comps, tem_ciclo

def validar_ciclo(g, ciclo, esperado):
    assert (ciclo is not None) == esperado, 'Detecção de ciclo incorreta.'
    if ciclo is None: return
    assert len(ciclo) >= 4 and ciclo[0] == ciclo[-1]
    assert len(set(ciclo[:-1])) == len(ciclo) - 1
    assert all(v in g.vizinhos(u) for u, v in zip(ciclo, ciclo[1:]))

def validar_dfs(g, r, comps, ciclo):
    vs = set(g.vertices)
    assert isinstance(r, ResultadoDFS)
    assert set(r.descoberta) == set(r.termino) == set(r.pais) == set(r.raiz_de) == vs
    assert len(r.ordem_descoberta) == len(r.ordem_termino) == len(vs)
    assert set(r.ordem_descoberta) == set(r.ordem_termino) == vs
    assert set(r.descoberta.values()) | set(r.termino.values()) == set(range(1, 2*len(vs)+1))
    assert list(r.ordem_descoberta) == sorted(vs, key=r.descoberta.get)
    assert list(r.ordem_termino) == sorted(vs, key=r.termino.get)
    assert {frozenset(c) for c in r.componentes} == comps
    assert tuple(v for v in r.ordem_descoberta if r.pais[v] is None) == r.raizes
    for u in vs:
        assert r.descoberta[u] < r.termino[u]
        p = r.pais[u]
        if p is not None:
            assert p in g.vizinhos(u)
            assert r.descoberta[p] < r.descoberta[u] < r.termino[u] < r.termino[p]
        assert r.raiz_de[u] in r.raizes
        assert r.raiz_de[u] == (u if p is None else r.raiz_de[p])
    for u, v in g.arestas:
        du, fu, dv, fv = r.descoberta[u], r.termino[u], r.descoberta[v], r.termino[v]
        assert du < dv < fv < fu or dv < du < fu < fv, 'Aresta incompatível com DFS não dirigida.'
    m = r.metricas
    assert m.vertices_descobertos == m.insercoes == m.remocoes == len(vs)
    assert m.entradas_examinadas == 2*len(g.arestas)
    assert (m.max_fronteira == 0 if not vs else 1 <= m.max_fronteira <= len(vs))
    validar_ciclo(g, r.ciclo, ciclo)

def executar_testes():
    if not pronto():
        print('Implemente as classes indicadas antes de executar os testes.'); return
    def erro(tipo, f):
        try: f()
        except tipo: return
        raise AssertionError('Era esperada a exceção ' + tipo.__name__)
    # Contratos de dados e estruturas de apoio.
    for vs, es in [(['A','A'], []), (['A'], [('A','A')]), (['A'], [('A','B')]),
                   (['A','B'], [('A','B'),('B','A')]), ([''], [])]:
        erro(ValueError, lambda vs=vs, es=es: Grafo(vs, es))
    g = Grafo(['B','A'], [('B','A')])
    assert g.vertices == ('A','B') and g.arestas == (('A','B'),)
    assert g.vizinhos('A') == ('B',)
    erro(KeyError, lambda: g.vizinhos('X'))
    erro(ValueError, lambda: g.sem_arestas([('A','X')]))
    f = Fila(); f.inserir('A'); f.inserir('B')
    assert f.fotografia() == ('A','B') and f.remover() == 'A' and f.remover() == 'B' and f.vazia()
    erro(IndexError, f.remover)
    p = PilhaQuadros(); q = QuadroDFS('A'); p.empilhar(q)
    foto = p.fotografia(); p.topo().proximo = 1
    assert q.proximo == 1 and foto == (('A',0),) and p.fotografia() == (('A',1),)
    assert p.desempilhar().vertice == 'A' and p.vazia()
    erro(IndexError, p.topo); erro(IndexError, p.desempilhar)
    print('1. Grafo, fila e pilha: contratos verificados.')
    # Exemplos das aulas e cenários independentes pequenos.
    g_bfs = Grafo(tuple('ABCDEFGH'), ARESTAS_AULA_BFS)
    r = BuscaLargura().executar(g_bfs, 'A')
    assert r.distancias == dict(zip('ABCDEFGH', [0,1,1,2,2,2,3,3]))
    assert r.caminho('H') == ('A','B','E','H')
    g_dfs = Grafo(tuple('ABCDEFGH'), ARESTAS_AULA_DFS)
    r = BuscaProfundidadeRecursiva().executar(g_dfs)
    assert r.ordem_descoberta == tuple('ABDECFHG')
    assert r.ordem_termino == tuple('EDBHFGCA')
    assert r.descoberta == dict(zip('ABCDEFGH', [1,2,8,3,4,9,13,10]))
    assert r.termino == dict(zip('ABCDEFGH', [16,7,15,6,5,12,14,11]))
    redes = [Grafo([], []), Grafo(['A'], []), g_bfs, g_dfs,
             Grafo(tuple('ABCD'), []), Grafo(tuple('ABCD'), [('A','B'),('C','D')])]
    rng = random.Random(503)
    for n in range(2, 11):
        for prob in (0.12, 0.4, 0.85):
            vs = [f'N{i:02}' for i in range(n)]
            es = [(u,v) for i,u in enumerate(vs) for v in vs[i+1:] if rng.random() < prob]
            redes.append(Grafo(vs, es))
    consultas = 0
    for rede in redes:
        ds, cs, cyc = referencias_independentes(rede)
        for estrategia in (BuscaProfundidadeRecursiva(), BuscaProfundidadeIterativa()):
            validar_dfs(rede, estrategia.executar(rede), cs, cyc)
        a = BuscaProfundidadeRecursiva().executar(rede)
        b = BuscaProfundidadeIterativa().executar(rede)
        for campo in ('pais','descoberta','termino','raiz_de','raizes','ordem_descoberta','ordem_termino','ciclo','metricas'):
            assert getattr(a, campo) == getattr(b, campo), campo
        for origem in rede.vertices:
            bfs = BuscaLargura().executar(rede, origem)
            assert isinstance(bfs, ResultadoBFS)
            assert bfs.origem == origem and len(bfs.ordem) == len(set(bfs.ordem))
            alc = [v for v in rede.vertices if not math.isinf(ds[origem,v])]
            assert set(bfs.ordem) == set(alc)
            m = bfs.metricas
            assert m.vertices_descobertos == m.insercoes == m.remocoes == len(alc)
            assert m.entradas_examinadas == sum(len(rede.vizinhos(v)) for v in alc)
            assert 1 <= m.max_fronteira <= len(alc)
            for destino in rede.vertices:
                consultas += 1
                assert bfs.distancias[destino] == ds[origem,destino]
                caminho = bfs.caminho(destino)
                if math.isinf(ds[origem,destino]): assert caminho is None
                else:
                    assert caminho[0] == origem and caminho[-1] == destino
                    assert len(caminho)-1 == ds[origem,destino]
                    assert all(v in rede.vizinhos(u) for u,v in zip(caminho,caminho[1:]))
            erro(KeyError, lambda: bfs.caminho('INEXISTENTE'))
    erro(ValueError, lambda: BuscaLargura().executar(g_bfs))
    for cl in (BuscaLargura, BuscaProfundidadeRecursiva, BuscaProfundidadeIterativa):
        assert issubclass(cl, EstrategiaPercurso)
        erro(KeyError, lambda cl=cl: cl().executar(g_bfs, 'INEXISTENTE'))
        obj = cl(); original = obj.executar(g_bfs, 'A')
        copia = repr(original)
        obj.executar(Grafo(['Z'], []), 'Z')
        assert repr(original) == copia, 'Resultado anterior foi alterado por nova execução.'
    print(f'2. {len(redes)} grafos, {consultas} consultas origem-destino e invariantes DFS: corretos.')
    # Delegação observável para as estruturas e para as estratégias do serviço.
    instancias_fila = []
    class FilaEspia(Fila):
        def __init__(self):
            super().__init__(); self.inseridos=0; self.retirados=0; instancias_fila.append(self)
        def inserir(self, v): self.inseridos += 1; return super().inserir(v)
        def remover(self): self.retirados += 1; return super().remover()
    instancias_pilha = []
    class PilhaEspia(PilhaQuadros):
        def __init__(self):
            super().__init__(); self.empilhados=0; self.retirados=0; self.topos=0; instancias_pilha.append(self)
        def empilhar(self, q): self.empilhados += 1; return super().empilhar(q)
        def desempilhar(self): self.retirados += 1; return super().desempilhar()
        def topo(self): self.topos += 1; return super().topo()
    BuscaLargura(fabrica_fila=FilaEspia).executar(g_bfs, 'A')
    BuscaProfundidadeIterativa(fabrica_pilha=PilhaEspia).executar(g_bfs)
    assert sum(x.inseridos for x in instancias_fila) == 8
    assert sum(x.retirados for x in instancias_fila) == 8
    assert sum(x.empilhados for x in instancias_pilha) == 8
    assert sum(x.retirados for x in instancias_pilha) == 8 and sum(x.topos for x in instancias_pilha) > 0
    class EstrategiaEspia(EstrategiaPercurso):
        def __init__(self, interna): self.interna=interna; self.chamadas=[]
        def executar(self, grafo, origem=None):
            self.chamadas.append((grafo, origem)); return self.interna.executar(grafo, origem)
    lb = EstrategiaEspia(BuscaLargura()); dp = EstrategiaEspia(BuscaProfundidadeIterativa())
    serv = ServicoInspecao(g_bfs, lb, dp)
    assert serv.rota('A','H') == ('A','B','E','H') and len(lb.chamadas) == 1
    assert isinstance(serv.diagnosticar(), ResultadoDFS) and len(dp.chamadas) == 1
    rel = serv.avaliar_interdicao([('A','B'),('A','C')], 'A','H')
    assert len(lb.chamadas) == 3 and len(dp.chamadas) == 3
    assert rel['rota_depois'] is None and rel['componentes_antes'] == 1 and rel['componentes_depois'] == 2
    assert rel['novos_inacessiveis'] == tuple('BCDEFGH')
    assert len(g_bfs.arestas) == len(ARESTAS_AULA_BFS), 'Interdição alterou o grafo original.'
    for estr in (BuscaProfundidadeRecursiva(), BuscaProfundidadeIterativa()):
        r = estr.executar(g_bfs, 'H'); assert r.raizes[0] == 'H'
    print('3. Herança, composição, estruturas injetadas, polimorfismo e independência: verificados.')
    a = BuscaProfundidadeRecursiva(registrar=True).executar(g_bfs)
    b = BuscaProfundidadeIterativa(registrar=True).executar(g_bfs)
    assert a.eventos == b.eventos, 'Os rastros das duas DFS divergem.'
    for r in (a,b):
        assert len(r.eventos) == 16
        assert tuple(dict(e.estados)[e.vertice] for e in r.eventos if e.tipo=='finalizar') == ('F',)*8
        assert r.eventos[-1].fronteira == ()
    rb = BuscaLargura(registrar=True).executar(g_bfs,'A')
    assert len(rb.eventos) == 16 and rb.eventos[0].fronteira == ('A',)
    assert all(dict(e.estados)[e.vertice] == 'E' for e in rb.eventos if e.tipo=='descobrir')
    assert rb.eventos[-1].fronteira == ()
    print('4. Rastros FIFO/LIFO, três estados e equivalência entre eventos: corretos.')
    print('Todos os testes fornecidos passaram. A estrutura e a complexidade também serão revisadas no código.')

executar_testes()


Implemente as classes indicadas antes de executar os testes.


### Dois testes adicionais de sua autoria

Adicione dois testes que investiguem riscos diferentes e explique o erro que cada um procura detectar. Não copie apenas os casos fornecidos. Sugestões: comparar fila larga e pilha rasa em uma estrela; excluir da lista de novos afetados sensores que já estavam isolados; confirmar que uma fotografia não muda após nova execução.


**Justificativa dos testes próprios**

_Escreva sua resposta aqui._


## 11. Demonstração industrial e visualizações prontas

A rede principal A–H é o grafo da aula de BFS. I–J constitui uma componente remota e K é isolado. O controlador de origem é A; o equipamento de destino é H.

Execute a célula. Ela usa **seus resultados e seus eventos**, sem recomputar a busca para desenhar. Por padrão, mostra quatro fotografias. Para inspecionar outra transição, chame `visualizar_eventos(grafo, resultado, indices=[...])` com índices válidos. Em BFS, a fotografia da fila vai do início ao fim. Em DFS, a pilha vai da base ao topo.

A célula também compara a rota de BFS com o caminho de pais da árvore DFS e avalia duas interdições: apenas E–H e os dois enlaces que saem de A.


In [ ]:
def visualizar_eventos(grafo, resultado, indices=None, titulo='Rastro do seu algoritmo'):
    if not resultado.eventos:
        print('Ative registrar=True para obter o rastro.'); return
    if indices is None:
        indices = sorted(set((0, min(3,len(resultado.eventos)-1), min(7,len(resultado.eventos)-1), len(resultado.eventos)-1)))
    vs = grafo.vertices
    pos = {v: (math.cos(2*math.pi*i/len(vs)), math.sin(2*math.pi*i/len(vs))) for i,v in enumerate(vs)}
    if vs == tuple('ABCDEFGH'):
        pos = {'A':(-1,0),'B':(-.5,.65),'C':(-.5,-.65),'D':(.05,.9),
               'E':(.05,.05),'F':(.05,-.9),'G':(.9,.65),'H':(.9,-.65)}
    fig, axs = plt.subplots(1, len(indices), figsize=(4.5*len(indices), 5.1), squeeze=False)
    for ax, idx in zip(axs[0], indices):
        e = resultado.eventos[idx]; estados=dict(e.estados)
        for u,v in grafo.arestas:
            ax.plot([pos[u][0],pos[v][0]],[pos[u][1],pos[v][1]], color='#b8bec9',zorder=1)
        for u in vs:
            cor={'N':'#ffffff','E':'#ffe2a8','F':'#26374c'}[estados[u]]
            ax.scatter(*pos[u],s=680,color=cor,edgecolors='#6857a6',linewidths=2,zorder=2)
            ax.text(*pos[u],u,ha='center',va='center',color='white' if estados[u]=='F' else '#17243a',zorder=3)
        fronteira = repr(e.fronteira)
        # Quebrar a representação evita uma linha larga para pilhas com muitos quadros.
        import textwrap
        ax.text(0.5,-0.06,'\n'.join(textwrap.wrap(fronteira,35)),transform=ax.transAxes,
                ha='center',va='top',fontsize=9)
        ax.set_title(f'{idx}: {e.tipo} {e.vertice}\nt={e.tempo}' if e.tempo is not None else f'{idx}: {e.tipo} {e.vertice}')
        ax.set_xlim(-1.3,1.25);ax.set_ylim(-1.25,1.25);ax.set_aspect('equal');ax.axis('off')
    fig.suptitle(titulo+'\nBranco: N; amarelo: E; escuro: F. BFS: início ao fim. DFS: base ao topo.',fontsize=12)
    fig.subplots_adjust(top=.78,bottom=.21,wspace=.25);plt.show()

def demonstracao():
    if not pronto(): print('Demonstração disponível após implementar as classes.'); return
    g = Grafo(tuple('ABCDEFGH'), ARESTAS_AULA_BFS)
    bfs = BuscaLargura(registrar=True).executar(g,'A')
    dfs = BuscaProfundidadeIterativa(registrar=True).executar(g)
    visualizar_eventos(g,bfs,titulo='BFS: fila após cada evento')
    visualizar_eventos(g,dfs,titulo='DFS iterativa: pilha de quadros após cada evento')
    print('BFS: caminho mínimo até H =',bfs.caminho('H'))
    inverso=[];u='H'
    while u is not None: inverso.append(u);u=dfs.pais[u]
    print('DFS: caminho da árvore até H =',tuple(reversed(inverso)))
    rede = Grafo(tuple('ABCDEFGHIJK'), ARESTAS_AULA_BFS + (('I','J'),))
    serv = ServicoInspecao(rede, BuscaLargura(), BuscaProfundidadeIterativa())
    print('Componentes da fábrica:',serv.diagnosticar().componentes)
    print('Interdição somente E–H:',serv.avaliar_interdicao([('E','H')],'A','H'))
    print('Isolamento do controlador A:',serv.avaliar_interdicao([('A','B'),('A','C')],'A','H'))

demonstracao()


Demonstração disponível após implementar as classes.


**G1 — Rotas, ciclos e interdições**

_Escreva sua resposta aqui._


## 12. Experimentos controlados

Compare cadeias e grades com 64, 144 e 400 vértices. Os grafos são conexos, de modo que BFS e DFS examinam todos os vértices e a comparação de operações cobre o mesmo conjunto de dados.

O código pronto mede três execuções após aquecimento e apresenta a mediana. Construção e ordenação do grafo, desenho, impressão e validação ficam fora da medição. Os resultados e as métricas continuam incluídos no tempo de `executar`. O registro de eventos fica desativado. A DFS recursiva só é medida até 196 vértices para reduzir dependência do limite da sessão. Não altere `sys.setrecursionlimit`.

Depois, a célula executa **somente a DFS iterativa** em uma cadeia de 3.000 vértices. Isso demonstra uma opção para grafos profundos, sem provocar intencionalmente erro de recursão.

Responda:

1. Por que o total de entradas examinadas é `2|E|` nos cenários conexos, ainda que a ordem de visita mude?
2. Como a forma do grafo afeta o tamanho máximo da fila e da pilha? Use também seu teste da estrela.
3. A igualdade entre pais e tempos das duas DFS implica igualdade de tempo de execução? Por quê?
4. As medições permitem provar Θ(|V|+|E|)? Diferencie contagem estrutural e evidência experimental.
5. Qual custo extra o registro de eventos introduz? Como ele distorceria a medição?


In [ ]:
def gerar_familia(tipo, tamanho):
    if tipo == 'cadeia':
        vs = [f'V{i:05}' for i in range(tamanho)]
        es = list(zip(vs, vs[1:]))
    elif tipo == 'grade':
        lado = math.isqrt(tamanho)
        if lado*lado != tamanho: raise ValueError('A grade exige quadrado perfeito.')
        vs = [f'V{i:05}' for i in range(tamanho)]
        es=[]
        for i in range(lado):
            for j in range(lado):
                k=i*lado+j
                if j+1<lado: es.append((vs[k],vs[k+1]))
                if i+1<lado: es.append((vs[k],vs[k+lado]))
    else: raise ValueError(tipo)
    return Grafo(vs, es)

def executar_experimentos():
    if not pronto(): print('Experimentos disponíveis após implementar as classes.'); return None
    linhas=[]
    for tipo in ('cadeia','grade'):
        for n in (64,144,400):
            g=gerar_familia(tipo,n)  # preparação e ordenação fora da medição
            for nome, classe in [('BFS',BuscaLargura),('DFS recursiva',BuscaProfundidadeRecursiva),
                                 ('DFS iterativa',BuscaProfundidadeIterativa)]:
                if nome == 'DFS recursiva' and n > 196:
                    continue  # evita depender do limite de recursão da sessão
                estrategia=classe(registrar=False)
                estrategia.executar(g,g.vertices[0])  # aquecimento
                tempos=[]
                for _ in range(3):
                    inicio=perf_counter();r=estrategia.executar(g,g.vertices[0]);tempos.append(perf_counter()-inicio)
                linhas.append({'familia':tipo,'n':n,'arestas':len(g.arestas),'metodo':nome,
                               'tempo_ms':1000*statistics.median(tempos),
                               'entradas_examinadas':r.metricas.entradas_examinadas,
                               'max_fronteira':r.metricas.max_fronteira,
                               'razao_operacoes':r.metricas.entradas_examinadas/(n+len(g.arestas))})
    df=pd.DataFrame(linhas)
    print(df.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
    fig,axs=plt.subplots(1,2,figsize=(12,4.8))
    for (tipo,nome),grupo in df.groupby(['familia','metodo']):
        axs[0].plot(grupo.n,grupo.entradas_examinadas,marker='o',label=f'{tipo}: {nome}')
        axs[1].plot(grupo.n,grupo.max_fronteira,marker='o',label=f'{tipo}: {nome}')
    axs[0].set(title='Trabalho contado',xlabel='Vértices',ylabel='Entradas de adjacência examinadas')
    axs[1].set(title='Memória da fronteira',xlabel='Vértices',ylabel='Máximo de elementos na fila/pilha')
    for ax in axs: ax.grid(alpha=.2)
    axs[1].legend(fontsize=8);fig.tight_layout();plt.show()
    fig,axs=plt.subplots(1,2,figsize=(12,4.5))
    for ax,tipo in zip(axs,('cadeia','grade')):
        for nome,grupo in df[df.familia==tipo].groupby('metodo'):
            ax.plot(grupo.n,grupo.tempo_ms,marker='o',label=nome)
        ax.set(title=f'Tempo: {tipo}',xlabel='Vértices',ylabel='Mediana de 3 execuções (ms)');ax.grid(alpha=.2);ax.legend()
    fig.tight_layout();plt.show()
    # Executar somente a versão iterativa em uma cadeia profunda.
    g=gerar_familia('cadeia',3000)
    r=BuscaProfundidadeIterativa().executar(g)
    assert len(r.ordem_descoberta)==3000 and r.metricas.max_fronteira==3000
    assert r.metricas.entradas_examinadas==5998
    print('Cadeia profunda: 3.000 vértices, 2.999 enlaces e pilha máxima de 3.000 quadros. DFS iterativa concluída.')
    print('Não executamos a versão recursiva nesse caso e não alteramos sys.setrecursionlimit.')
    return df

resultados_experimentos = executar_experimentos()


Experimentos disponíveis após implementar as classes.


**H1 — Interpretação dos experimentos**

_Escreva sua resposta aqui._


## 13. Parecer técnico e entrega

Escreva **400–600 palavras** como responsável técnico pelo serviço de inspeção. Use pelo menos três valores retirados de suas tabelas ou métricas, cubra os dois cenários de interdição e apresente:

- qual algoritmo atende às rotas mínimas e qual atende ao diagnóstico topológico;
- a escolha entre recursão e pilha explícita para históricos/topologias profundos;
- a função concreta da herança, da composição e do encapsulamento no seu sistema;
- o significado e as limitações de um certificado de ciclo;
- os limites da modelagem sem pesos e da evidência sintética.

Não conclua que uma rota é fisicamente segura nem que o número de saltos mede latência. Antes de entregar: preencha identificação, implemente todas as classes, escreva os dois testes próprios e as respostas, reinicie e execute em ordem, preserve as saídas e faça download do `.ipynb`. Registre o uso de ferramentas externas conforme a política da disciplina.


**Parecer técnico**

_Escreva sua resposta aqui._


## Referências e ligação com as aulas

- **SSC0503_Busca_em_Largura.pdf**, material fornecido pelo professor: camadas e FIFO (páginas 4–5), exemplo e predecessores (7–12), pseudocódigo (13), componentes e complexidade (15–16). O arquivo tem 17 páginas, embora a numeração visual cite um conjunto de 18.
- **Busca_em_Profundidade.pdf**, material fornecido pelo professor: estados e exemplo (páginas 5–25), recursão e quadros (26–32), distinção da variante que empilha só vértices (33), complexidade e conferência (34–36). A numeração indicada aqui é a página do PDF, não os contadores antigos que permanecem em alguns slides.
- Cormen et al. *Introduction to Algorithms*, 4ª ed., MIT Press, 2022, capítulo 20.
- Sedgewick e Wayne. *Algorithms*, 4ª ed., Addison-Wesley, 2011, seção 4.1.

O notebook usa os dois grafos das aulas como casos separados de validação. A aplicação industrial e os experimentos são sintéticos. Pseudocódigos adaptados aos contratos desta atividade.
